In [5]:
import pandas as pd
import warnings

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, StandardScaler

from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score
)

warnings.filterwarnings("ignore")

# ==========================
# Load Dataset

# ==========================

df = pd.read_csv("balanced_dataset.csv")

# ==========================
# Encode categorical columns
# ==========================

df_ml = df.copy()

for col in df_ml.select_dtypes(include="object").columns:
    encoder = LabelEncoder()
    df_ml[col] = encoder.fit_transform(df_ml[col])

# ==========================
# Features and Target
# ==========================

X = df_ml.drop("Diabetes_012", axis=1)
y = df_ml["Diabetes_012"]

# ==========================
# Train-Test Split
# ==========================

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

# ==========================
# Feature Scaling
# (Only for Logistic Regression)
# ==========================

scaler = StandardScaler()

X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# ==========================
# Models
# ==========================

models = {

    "Logistic Regression":
    (
        LogisticRegression(
            random_state=42,
            max_iter=1000
        ),
        X_train_scaled,
        X_test_scaled
    ),

    "Decision Tree":
    (
        DecisionTreeClassifier(
            random_state=42,
            max_depth=10
        ),
        X_train,
        X_test
    ),

    "Random Forest":
    (
        RandomForestClassifier(
            n_estimators=200,
            random_state=42,
            n_jobs=-1
        ),
        X_train,
        X_test
    ),

    "XGBoost":
    (
        XGBClassifier(
            n_estimators=200,
            learning_rate=0.05,
            max_depth=5,
            eval_metric="logloss",
            random_state=42,
            n_jobs=-1
        ),
        X_train,
        X_test
    )
}

# ==========================
# Model Evaluation
# ==========================

results = []

for name, (model, train_data, test_data) in models.items():

    print(f"Training {name}...")

    model.fit(train_data, y_train)

    pred = model.predict(test_data)

    prob = model.predict_proba(test_data)[:, 1]

    results.append({

        "Model": name,

        "Accuracy": accuracy_score(y_test, pred),

        "Precision": precision_score(
            y_test,
            pred,
            zero_division=0
        ),

        "Recall": recall_score(
            y_test,
            pred,
            zero_division=0
        ),

        "F1 Score": f1_score(
            y_test,
            pred,
            zero_division=0
        ),

        "ROC AUC": roc_auc_score(
            y_test,
            prob
        )

    })

# ==========================
# Results
# ==========================

results_df = pd.DataFrame(results)

results_df = results_df.sort_values(
    by="F1 Score",
    ascending=False
)

print("\nModel Comparison\n")

print(results_df)

Training Logistic Regression...
Training Decision Tree...
Training Random Forest...
Training XGBoost...

Model Comparison

                 Model  Accuracy  Precision    Recall  F1 Score   ROC AUC
3              XGBoost  0.728852   0.709787  0.774289  0.740636  0.803356
0  Logistic Regression  0.724622   0.716438  0.743527  0.729731  0.798796
1        Decision Tree  0.713022   0.705999  0.730069  0.717832  0.779592
2        Random Forest  0.697962   0.687189  0.726737  0.706410  0.757010
